# 03 - Sentiment Model: Data Loading & EDA

Women's E-Commerce Clothing Reviews dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/reviews.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
print("Rating value counts:")
print(df['Rating'].value_counts().sort_index())
print()
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(6,4))
df['Rating'].value_counts().sort_index().plot(kind='bar', color='seagreen')
plt.xlabel("Rating (1-5)")
plt.ylabel("Count")
plt.title("Rating distribution")
plt.tight_layout()
plt.savefig('rating_dist.png', dpi=100)
plt.show()

In [ ]:
def label_sentiment(r):
    if r <= 2: return 'Negative'
    elif r == 3: return 'Neutral'
    else: return 'Positive'

df['sentiment'] = df['Rating'].apply(label_sentiment)
print(df['sentiment'].value_counts())

In [ ]:
plt.figure(figsize=(5,4))
df['sentiment'].value_counts().plot(kind='bar', color=['seagreen','goldenrod','indianred'])
plt.title("Sentiment label distribution (derived from Rating)")
plt.tight_layout()
plt.savefig('sentiment_dist.png', dpi=100)
plt.show()

In [ ]:
df[['Review Text','Rating','sentiment']].dropna().sample(5, random_state=1)

**Observation:** Ratings are heavily skewed toward 4-5 stars, so the derived sentiment classes are imbalanced (mostly Positive). This needs class weighting or stratified sampling when training the TF-IDF + Logistic Regression sentiment model below — noting this now so it's not a surprise later.

## Text Preprocessing & Model Training

In [ ]:
import sys
sys.path.insert(0, '../app/services')
from nlp_service import preprocess

sample = df['Review Text'].dropna().iloc[1]
print("Before:", sample[:150])
print("After: ", preprocess(sample)[:150])

In [ ]:
df_clean = df.dropna(subset=['Review Text', 'Rating']).copy()
print("Preprocessing all reviews (takes ~1 min)...")
df_clean['clean_text'] = df_clean['Review Text'].apply(preprocess)
df_clean[['Review Text', 'clean_text', 'sentiment']].head(3)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df_clean['clean_text'], df_clean['sentiment'],
    test_size=0.2, random_state=42, stratify=df_clean['sentiment']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print("TF-IDF matrix shape:", X_train_tfidf.shape)

**Note:** `class_weight='balanced'` is used to counter the Positive-heavy skew identified in the EDA above — otherwise the model would just learn to predict "Positive" most of the time.

In [ ]:
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}\n")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Negative','Neutral','Positive'])
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks([0,1,2], ['Negative','Neutral','Positive'])
plt.yticks([0,1,2], ['Negative','Neutral','Positive'])
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i,j], ha='center', va='center')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Sentiment Confusion Matrix')
plt.tight_layout()
plt.savefig('sentiment_confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
import pickle
with open('../app/models/sentiment_model.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open('../app/models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("Saved sentiment_model.pkl and vectorizer.pkl")

**Result:** 76.9% test accuracy. As expected from the imbalance flagged in the EDA, the Positive class (majority) is predicted well (F1 ≈ 0.89), while Negative and Neutral are noticeably weaker (F1 ≈ 0.5, 0.4) — the model confuses these two more often, which makes sense since 3-star ("Neutral") reviews often contain a genuine mix of positive and negative language. Class weighting improved recall on the minority classes compared to an unweighted baseline, but did not eliminate the imbalance effect entirely — worth noting honestly in the report rather than only reporting overall accuracy, which would overstate performance on the minority classes.